# Analytical Benchmark Case

Minimal benchmark notebook using the NF2 Python API for one analytical case.

In [ ]:
from pathlib import Path

RUN_TRAINING = True
CASE = 1
RUN_ROOT = Path("runs/benchmark")
EXPORT_MM_PER_PIXEL = 0.05
HEIGHT_MIN = 0
HEIGHT_MAX = 2
EPOCHS = 10
VALIDATION_PIXEL_PER_DS = 32
CASE_RESOLUTION = {1: 64, 2: [80, 80, 72]}

In [ ]:
import os
os.environ.setdefault("WANDB_CONSOLE", "off")

from pathlib import Path
from dateutil.parser import parse

import matplotlib.pyplot as plt
import numpy as np
import wandb

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm

from nf2.data.analytical_field import get_analytic_b_field
from nf2.evaluation.metric import evaluate

In [ ]:
run_dir = RUN_ROOT / f"case{CASE}"
config = {
    "path": str(run_dir),
    "work_path": str(run_dir / "work"),
    "logging": {"project": "nf2", "name": f"Analytical case {CASE}"},
    "model": {"network": {"type": "siren", "w0_init": 1}},
    "data": {
        "geometry": "cartesian",
        "normalization": {"Mm_per_ds": 1, "Gauss_per_dB": 1},
        "boundaries": [
            {
                "id": "boundary",
                "type": "analytical",
                "case": CASE,
                "boundary": "full",
                "resolution": CASE_RESOLUTION[CASE],
                "batch_size": 4096,
            }
        ],
        "sampler": {"type": "height", "batch_size": 8192},
        "potential_boundary": {"type": "none"},
        "validation": [
            {
                "id": "boundary",
                "type": "analytical",
                "case": CASE,
                "boundary": "bottom",
                "resolution": CASE_RESOLUTION[CASE],
                "batch_size": 4096,
            },
            {"id": "slices", "type": "slices", "n_slices": 5, "batch_size": 8192},
        ],
        "iterations": 1000,
        "z_range": [HEIGHT_MIN, HEIGHT_MAX],
        "validation_pixel_per_ds": VALIDATION_PIXEL_PER_DS,
    },
    "training": {"epochs": EPOCHS, "optimizer": 1.0e-4},
    "losses": [
        {"type": "boundary", "name": "boundary", "weight": 1.0, "datasets": "boundary"},
        {"type": "force_free", "name": "force_free", "weight": 1.0e-3, "datasets": ["random"]},
    ],
    "loss_scaling": [],
    "callbacks": [
        {"type": "boundary", "dataset": "boundary"},
        {"type": "slices", "dataset": "slices"},
    ],
}
if RUN_TRAINING:
    nf2.run(**config)
    wandb.finish()
else:
    config
NF2_PATH = run_dir / "extrapolation_result.nf2"
NF2_PATH

In [ ]:
EXPORT_DIR = NF2_PATH.parent / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / f"case{CASE}.vtk"), fmt="vtk", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy"])

out = nf2.load(NF2_PATH)
cube = out.load_cube(Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "j_vec", "free_energy"], progress=True)

def values(x):
    return np.asarray(getattr(x, "value", x))

b = values(cube["b"])
j = values(cube["metrics"]["j"])
j_vec = values(cube["metrics"]["j_vec"])
free_energy = values(cube["metrics"]["free_energy"])
reference = get_analytic_b_field(n=1, m=1, l=0.3, psi=np.pi / 4 if CASE == 1 else np.pi * 0.15, resolution=list(b.shape[:3]), bounds=[-1, 1, -1, 1, 0, 2])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j_vec)),
    "sigma_J_1e2": float(sigma_J(b, j_vec) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(j)),
    "total_free_energy_density": float(np.nansum(free_energy)),
    **{f"reference_{k}": float(v) for k, v in evaluate(b, reference).items() if np.isscalar(v)},
}
metrics

In [ ]:
out = nf2.load(NF2_PATH)
cube = out.load_cube(Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "j_vec", "free_energy"], progress=True)
b = np.asarray(getattr(cube["b"], "value", cube["b"]))
j = np.asarray(getattr(cube["metrics"]["j"], "value", cube["metrics"]["j"]))
j_vec = np.asarray(getattr(cube["metrics"]["j_vec"], "value", cube["metrics"]["j_vec"]))
free_energy = np.asarray(getattr(cube["metrics"]["free_energy"], "value", cube["metrics"]["free_energy"]))

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, image, title, cmap in [
    (axs[0], np.nansum(j, axis=2).T, "Integrated |J|", "magma"),
    (axs[1], np.nansum(free_energy, axis=2).T, "Free energy", "inferno"),
    (axs[2], b[:, :, 0, 2].T, "Model boundary Bz", "RdBu_r"),
]:
    kwargs = {"origin": "lower", "cmap": cmap}
    if cmap == "RdBu_r":
        lim = np.nanpercentile(np.abs(image), 99)
        kwargs.update(vmin=-lim, vmax=lim)
    im = ax.imshow(image, **kwargs)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()